# Reference Loader

The `reference_loader` method downloads reference data files from a HTTPS source, stores the raw file in the Lakehouse, loads the data into a Bronze Delta table, and writes an ingestion log.

## Functionality

The loader performs the following steps:

1. Downloads the file from the provided URL.
2. Handles UTF-8 encoded files, including files with a UTF-8 BOM.
3. Stores the raw file in: /lakehouse/default/Files/data/raw
4. Loads the CSV file into a Spark DataFrame.
5. Writes the data as a Delta table in the `bronze` schema.
6. Records ingestion details in the logging framework, including:
- source name
- file name
- URL
- ingestion timestamps
- download and write duration
- file size
- row count
- success/failure status
- error details

## Usage

The method can be called from another notebook using `%run`:

```python
%run reference_loader

reference_loader(
 "airports",
 "https://ourairports.com/data/airports.csv"
)

In [1]:
%run log

StatementMeta(, a0a925d6-9967-42d5-bdfe-de8bada1be84, 3, Finished, Available, Finished, True)

In [ ]:
# reference_loader
# https://ourairports.com/data/airports.csv
# https://raw.githubusercontent.com/benct/iata-utils/master/generated/iata_airlines.csv


# Method that takes a https://-link and download the file and returns it with stats for logging.
# =========================================================================

import os    
import requests
from urllib.parse import urlparse
from pathlib import Path
from datetime import datetime, timezone

UTF8_BOM = b"\xef\xbb\xbf" # Sååå trött på byte-order trams...

def download_file(url):
    try:
        print(f"Trying to download from: {url}")
        response = requests.get(url, timeout=600)
        response.raise_for_status()

        content_bytes = response.content
        has_utf8_bom = content_bytes.startswith(UTF8_BOM)

        if has_utf8_bom:
            content = content_bytes.decode("utf-8-sig")
        else:
            content = content_bytes.decode("utf-8")

        return {
            "status": "SUCCESS",
            "status_code": response.status_code,
            "content": content,
            "content_bytes" : len(content_bytes),
            "has_utf8_bom": has_utf8_bom,
            "error_message": None
        }

    except (requests.RequestException, UnicodeDecodeError) as error:
        return {
            "status": "FAILED",
            "status_code": (
                error.response.status_code
                if isinstance(error, requests.RequestException)
                and error.response is not None
                else None
            ),
            "content": None,
            "content_bytes" : 0,
            "has_utf8_bom": None,
            "error_message": str(error)
        }

def reference_loader(source_name, url):    
    ingestion_start_time = datetime.now(timezone.utc)    

    file_name = Path(urlparse(url).path).name
    root_path = "/lakehouse/default"
    folder_path = "Files/data/raw"    
    file_path = f"{root_path}/{folder_path}/{file_name}"

    table_name = Path(file_name).stem

    os.makedirs(f"{root_path}/{folder_path}", exist_ok=True)    

    start_download = datetime.now(timezone.utc)
    dl_result = download_file(url)
    download_time = datetime.now(timezone.utc) - start_download

    dl_success = dl_result["status"]
    file_content = dl_result["content"]
    dl_error = dl_result["error_message"]

    if dl_success != "SUCCESS":
        ingestion_stop_time = datetime.now(timezone.utc)

        log_path = write_log(
            source_name,
            file_name,
            url,
            f"{folder_path}/{file_name}",
            table_name,
            "reference_loader",
            ingestion_start_time,
            ingestion_stop_time,
            download_time,
            None,       # write_time
            0,          # size_bytes
            0,          # rows
            dl_success,
            dl_error,
            "json",
            "/lakehouse/default/Files/data/log"
        )

        print(f"Download failed: {dl_error}")
        print(f"Log was written to {log_path}")
        return

    size_bytes = dl_result["content_bytes"]

    start_write = datetime.now(timezone.utc)

    with open(file_path, "w", encoding="utf-8", newline="") as file:
        file.write(file_content)

    print(f"Saved {file_name} to {file_path}")

    spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
    df = spark.read.format("csv").option("header","true").load(f"{folder_path}/{file_name}")    
    (
        df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(f"bronze.{table_name}")
    )
    write_time = datetime.now(timezone.utc) - start_write
    rows = df.count()
    print(f"Rows loaded : {rows}")
    
    ingestion_stop_time = datetime.now(timezone.utc)

    log_path = write_log(
        source_name,
        file_name,
        url,
        f"{folder_path}/{file_name}",
        table_name,
        "reference_loader",
        ingestion_start_time,
        ingestion_stop_time,
        download_time,
        write_time,
        size_bytes,
        rows,        
        dl_success, # TODO: needs to also account for write errors
        dl_error, # TODO: needs to be cumulative error for dl and wr
        "json",
        "/lakehouse/default/Files/data/log"
    )  

    print(f"Log was written to {log_path}")       

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can't be run because you've hit spark overall capacity compute limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.